In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def make_onehot_encoder():
    """
    Compatible with both older and newer sklearn versions.
    Random Forest works better with dense output.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def prepare_diabetes_data(filepath):
    df = pd.read_csv(filepath)

    df = df.replace("?", np.nan)

    drop_cols = [
        "encounter_id",
        "patient_nbr",
    ]
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols)

    sparse_optional = ["weight", "payer_code", "medical_specialty"]
    sparse_existing = [c for c in sparse_optional if c in df.columns]
    df = df.drop(columns=sparse_existing)

    if "readmitted" not in df.columns:
        raise ValueError("Expected column 'readmitted' not found in diabetes dataset.")

    df["target"] = (df["readmitted"] == "<30").astype(int)
    df = df.drop(columns=["readmitted"])

    numeric_candidates = [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses"
    ]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    x = df.drop(columns=["target"])
    y = df["target"]

    return x, y


def prepare_german_credit_data(filepath):
    col_names = [
        "status", "duration", "credit_history", "purpose", "credit_amount",
        "savings", "employment", "installment_rate", "personal_status_sex",
        "other_debtors", "residence_since", "property", "age",
        "other_installment_plans", "housing", "existing_credits",
        "job", "num_dependents", "telephone", "foreign_worker", "target"
    ]

    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=col_names)

    # Convert target: 1 = good, 2 = bad -> make 1 = bad
    df["target"] = (df["target"] == 2).astype(int)

    # Extract gender for fairness analysis
    def extract_gender(val):
        if val in ["A91", "A93", "A94"]:
            return "male"
        elif val in ["A92", "A95"]:
            return "female"
        else:
            return "unknown"

    df["gender"] = df["personal_status_sex"].apply(extract_gender)

    x = df.drop(columns=["target"])
    y = df["target"]

    return x, y


def prepare_synthetic_diabetes_data(filepath):
    df = pd.read_csv(filepath)

    df = df.replace("?", np.nan)

    drop_cols = ["encounter_id", "patient_nbr"]
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols)

    sparse_optional = ["weight", "payer_code", "medical_specialty"]
    sparse_existing = [c for c in sparse_optional if c in df.columns]
    df = df.drop(columns=sparse_existing)

    if "target" not in df.columns:
        if "readmitted" not in df.columns:
            raise ValueError("Synthetic diabetes file must contain either 'target' or 'readmitted'.")
        df["target"] = (df["readmitted"] == "<30").astype(int)
        df = df.drop(columns=["readmitted"])
    else:
        df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)

    numeric_candidates = [
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses"
    ]
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def prepare_synthetic_german_credit_data(filepath):
    df = pd.read_csv(filepath)

    # Rename columns to match real dataset
    rename_map = {
        "checking_status": "status",
        "savings_status": "savings",
        "personal_status": "personal_status_sex",
        "installment_commitment": "installment_rate",
        "other_parties": "other_debtors",
        "property_magnitude": "property",
        "other_payment_plans": "other_installment_plans",
        "own_telephone": "telephone"
    }

    df = df.rename(columns=rename_map)

    if "target" in df.columns:
        unique_vals = set(df["target"].dropna().unique())
        if unique_vals.issubset({1, 2}):
            df["target"] = (df["target"] == 2).astype(int)
        else:
            df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype(int)

    elif "class" in df.columns:
        df["target"] = (df["class"] == 2).astype(int)
        df = df.drop(columns=["class"])

    else:
        raise ValueError("No target column found.")

    if "gender" not in df.columns and "personal_status_sex" in df.columns:
        def extract_gender(val):
            if val in ["A91", "A93", "A94"]:
                return "male"
            elif val in ["A92", "A95"]:
                return "female"
            else:
                return "unknown"

        df["gender"] = df["personal_status_sex"].apply(extract_gender)

    return df


def build_preprocessor_rf(X):
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numerical_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numerical_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    return preprocessor, numerical_cols, categorical_cols


def build_random_forest_model(preprocessor):
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])
    return model


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    num_unique_classes = len(np.unique(y_test))

    y_prob = None
    if hasattr(model, "predict_proba") and num_unique_classes > 1:
        probabilities = model.predict_proba(X_test)
        if probabilities.shape[1] > 1:
            y_prob = probabilities[:, 1]

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }

    if y_prob is not None and num_unique_classes > 1:
        results["auc"] = roc_auc_score(y_test, y_prob)
    else:
        results["auc"] = np.nan

    return results, y_pred, y_prob


def fairness_by_group(model, X_test, y_test, group_col):
    if group_col not in X_test.columns:
        raise ValueError(f"Column '{group_col}' not found in X_test.")

    temp = X_test.copy()
    temp["y_true"] = y_test.values
    temp["y_pred"] = model.predict(X_test)

    rows = []

    for group_value in temp[group_col].dropna().unique():
        g = temp[temp[group_col] == group_value]

        y_true_g = g["y_true"]
        y_pred_g = g["y_pred"]

        tn, fp, fn, tp = confusion_matrix(
            y_true_g, y_pred_g, labels=[0, 1]
        ).ravel()

        positive_rate = (y_pred_g == 1).mean()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

        rows.append({
            "group_column": group_col,
            "group_value": group_value,
            "n": len(g),
            "positive_rate": positive_rate,
            "FPR": fpr,
            "FNR": fnr,
            "precision": precision
        })

    return pd.DataFrame(rows)


def run_real_baseline_rf(X, y, dataset_name):
    X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
        X, y,
        test_size=0.5,
        random_state=42,
        stratify=y
    )

    preprocessor, numerical_cols, categorical_cols = build_preprocessor_rf(X_train_real)
    model = build_random_forest_model(preprocessor)

    model.fit(X_train_real, y_train_real)
    metrics, y_pred, y_prob = evaluate_model(model, X_test_real, y_test_real)

    print(f"\n=== {dataset_name}: REAL baseline (Random Forest) ===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("F1:", round(metrics["f1"], 4))
    print("AUC:", round(metrics["auc"], 4) if pd.notnull(metrics["auc"]) else "nan")

    return {
        "model": model,
        "X_train_real": X_train_real,
        "X_test_real": X_test_real,
        "y_train_real": y_train_real,
        "y_test_real": y_test_real,
        "metrics": metrics,
        "numerical_cols": numerical_cols,
        "categorical_cols": categorical_cols
    }


def run_synthetic_tstr_rf(
    synthetic_train_df,
    X_test_real,
    y_test_real,
    target_col="target"
):
    if target_col not in synthetic_train_df.columns:
        raise ValueError(f"Synthetic dataframe must contain target column '{target_col}'.")

    X_train_synth = synthetic_train_df.drop(columns=[target_col])
    y_train_synth = synthetic_train_df[target_col].astype(int)

    preprocessor, _, _ = build_preprocessor_rf(X_train_synth)
    model = build_random_forest_model(preprocessor)

    model.fit(X_train_synth, y_train_synth)
    metrics, y_pred, y_prob = evaluate_model(model, X_test_real, y_test_real)

    print("\n=== Synthetic Train / Real Test (Random Forest) ===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("F1:", round(metrics["f1"], 4))
    print("AUC:", round(metrics["auc"], 4) if pd.notnull(metrics["auc"]) else "nan")

    return {
        "model": model,
        "metrics": metrics
    }


def add_performance_result(rows, dataset_name, train_data_name, metrics_dict, test_data="Real", model_name="Random Forest"):
    rows.append({
        "Dataset": dataset_name,
        "Model": model_name,
        "Train Data": train_data_name,
        "Test Data": test_data,
        "Accuracy": round(metrics_dict["accuracy"], 4),
        "F1": round(metrics_dict["f1"], 4),
        "AUC": round(metrics_dict["auc"], 4) if pd.notnull(metrics_dict["auc"]) else None
    })


def add_fairness_result(rows, dataset_name, train_data_name, fairness_df, model_name="Random Forest"):
    for _, row in fairness_df.iterrows():
        rows.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Train Data": train_data_name,
            "Group Column": row["group_column"],
            "Group Value": row["group_value"],
            "N": int(row["n"]),
            "Positive Rate": round(row["positive_rate"], 4) if pd.notnull(row["positive_rate"]) else None,
            "FPR": round(row["FPR"], 4) if pd.notnull(row["FPR"]) else None,
            "FNR": round(row["FNR"], 4) if pd.notnull(row["FNR"]) else None,
            "Precision": round(row["precision"], 4) if pd.notnull(row["precision"]) else None
        })


In [9]:
# Real data
diab_real_df = prepare_synthetic_diabetes_data("/content/diabetic_data_synthetic_gen1.csv")
X_diab = diab_real_df.drop(columns=["target"])
y_diab = diab_real_df["target"]

credit_real_df = prepare_synthetic_german_credit_data("/content/german_credit_synthetic_gen1.csv")
X_credit = credit_real_df.drop(columns=["target"])
y_credit = credit_real_df["target"]

rf_diab_results = run_real_baseline_rf(
    X=X_diab,
    y=y_diab,
    dataset_name="UCI Diabetes"
)

rf_credit_results = run_real_baseline_rf(
    X=X_credit,
    y=y_credit,
    dataset_name="German Credit"
)


=== UCI Diabetes: REAL baseline (Random Forest) ===
Accuracy: 0.8903
F1: 0.0
AUC: 0.5391

=== German Credit: REAL baseline (Random Forest) ===
Accuracy: 0.704
F1: 0.0263
AUC: 0.4724


In [12]:
# Synthetic data
diab_synth_df = prepare_synthetic_diabetes_data("diabetic_data_synthetic_gen1.csv")
credit_synth_df = prepare_synthetic_german_credit_data("german_credit_synthetic_gen1.csv")

rf_diab_tstr = run_synthetic_tstr_rf(
    synthetic_train_df=diab_synth_df,
    X_test_real=rf_diab_results["X_test_real"],
    y_test_real=rf_diab_results["y_test_real"],
    target_col="target"
)

rf_credit_tstr = run_synthetic_tstr_rf(
    synthetic_train_df=credit_synth_df,
    X_test_real=rf_credit_results["X_test_real"],
    y_test_real=rf_credit_results["y_test_real"],
    target_col="target"
)



=== Synthetic Train / Real Test (Random Forest) ===
Accuracy: 1.0
F1: 1.0
AUC: 1.0

=== Synthetic Train / Real Test (Random Forest) ===
Accuracy: 1.0
F1: 1.0
AUC: 1.0


In [16]:
# Performance table
performance_rows_rf = []

add_performance_result(performance_rows_rf, "UCI Diabetes", "Real", rf_diab_results["metrics"])
add_performance_result(performance_rows_rf, "UCI Diabetes", "Synthetic", rf_diab_tstr["metrics"])

add_performance_result(performance_rows_rf, "German Credit", "Real", rf_credit_results["metrics"])
add_performance_result(performance_rows_rf, "German Credit", "Synthetic", rf_credit_tstr["metrics"])

performance_table_rf = pd.DataFrame(performance_rows_rf)
print("\n=== Random Forest Performance Table ===")
print(performance_table_rf)



=== Random Forest Performance Table ===
         Dataset          Model Train Data Test Data  Accuracy      F1     AUC
0   UCI Diabetes  Random Forest       Real      Real    0.8903  0.0000  0.5391
1   UCI Diabetes  Random Forest  Synthetic      Real    1.0000  1.0000  1.0000
2  German Credit  Random Forest       Real      Real    0.7040  0.0263  0.4724
3  German Credit  Random Forest  Synthetic      Real    1.0000  1.0000  1.0000


In [17]:
# Fairness tables
fairness_diab_real_gender_rf = fairness_by_group(
    rf_diab_results["model"],
    rf_diab_results["X_test_real"],
    rf_diab_results["y_test_real"],
    group_col="gender"
)

fairness_diab_real_race_rf = fairness_by_group(
    rf_diab_results["model"],
    rf_diab_results["X_test_real"],
    rf_diab_results["y_test_real"],
    group_col="race"
)

fairness_credit_real_gender_rf = fairness_by_group(
    rf_credit_results["model"],
    rf_credit_results["X_test_real"],
    rf_credit_results["y_test_real"],
    group_col="gender"
)

fairness_diab_synth_gender_rf = fairness_by_group(
    rf_diab_tstr["model"],
    rf_diab_results["X_test_real"],
    rf_diab_results["y_test_real"],
    group_col="gender"
)

fairness_diab_synth_race_rf = fairness_by_group(
    rf_diab_tstr["model"],
    rf_diab_results["X_test_real"],
    rf_diab_results["y_test_real"],
    group_col="race"
)

fairness_credit_synth_gender_rf = fairness_by_group(
    rf_credit_tstr["model"],
    rf_credit_results["X_test_real"],
    rf_credit_results["y_test_real"],
    group_col="gender"
)

fairness_rows_rf = []


In [18]:
# REAL
add_fairness_result(fairness_rows_rf, "UCI Diabetes", "Real", fairness_diab_real_gender_rf)
add_fairness_result(fairness_rows_rf, "UCI Diabetes", "Real", fairness_diab_real_race_rf)
add_fairness_result(fairness_rows_rf, "German Credit", "Real", fairness_credit_real_gender_rf)


In [19]:
# SYNTHETIC
add_fairness_result(fairness_rows_rf, "UCI Diabetes", "Synthetic", fairness_diab_synth_gender_rf)
add_fairness_result(fairness_rows_rf, "UCI Diabetes", "Synthetic", fairness_diab_synth_race_rf)
add_fairness_result(fairness_rows_rf, "German Credit", "Synthetic", fairness_credit_synth_gender_rf)

fairness_table_rf = pd.DataFrame(fairness_rows_rf)
fairness_table_rf["Group Value"] = fairness_table_rf["Group Value"].astype(str).str.lower()

print("\n=== Random Forest Fairness Table ===")
print(fairness_table_rf)



=== Random Forest Fairness Table ===
          Dataset          Model Train Data Group Column      Group Value  \
0    UCI Diabetes  Random Forest       Real       gender           female   
1    UCI Diabetes  Random Forest       Real       gender             male   
2    UCI Diabetes  Random Forest       Real       gender  unknown/invalid   
3    UCI Diabetes  Random Forest       Real         race        caucasian   
4    UCI Diabetes  Random Forest       Real         race  africanamerican   
5    UCI Diabetes  Random Forest       Real         race         hispanic   
6    UCI Diabetes  Random Forest       Real         race            other   
7    UCI Diabetes  Random Forest       Real         race            asian   
8   German Credit  Random Forest       Real       gender             male   
9   German Credit  Random Forest       Real       gender           female   
10   UCI Diabetes  Random Forest  Synthetic       gender           female   
11   UCI Diabetes  Random Forest  Synt